# Backtest Plotting Real Data Example

This notebook shows the canonical `CSVDataSource -> BacktestEngine.run(...) -> result.plot()` flow using the checked-in Binance USD-M BTC/USDT 1h 2025 fixture.


In [ ]:
from __future__ import annotations

from pathlib import Path

from quantcraft.backtest import BacktestEngine
from quantcraft.data import CSVDataSource
from quantcraft.research import Strategy, qc, ta
from quantcraft.trading.domain.costs import CostConfig


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("run this notebook from inside the quantcraft repository")


repo_root = find_repo_root(Path.cwd())
fixture_path = (
    repo_root
    / "tests"
    / "fixtures"
    / "backtest"
    / "binance_usdm_btcusdtusdt_1h_2025.csv"
)

source = CSVDataSource(
    path=fixture_path,
    symbol="BTC/USDT:USDT",
    timeframe="1h",
)

bars = source.load()
bars.symbol, bars.timeframe, len(bars.rows), bars.rows[0], bars.rows[-1]

The strategy below is the standard RSI 30/70 long-only example: buy when RSI falls below 30 and exit when RSI rises above 70.


In [ ]:
class Rsi3070Strategy(Strategy):
    def init(self) -> None:
        self.rsi = ta.rsi(self.data.close, length=14)

    def on_bar(self, bar) -> None:
        if qc.is_na(self.rsi[0]):
            return
        if (not self.position.is_open) and self.rsi[0] < 30:
            self.buy(quantity=1.0, tag="rsi-entry")
        elif self.position.is_open and self.rsi[0] > 70:
            self.sell(quantity=1.0, tag="rsi-exit")


engine = BacktestEngine(
    initial_cash=1_000_000.0,
    costs=CostConfig(tick_size=0.1, slippage_ticks=1.0, fee_rate=0.0004),
)

result = engine.run(
    source=source,
    strategy=Rsi3070Strategy(),
    label="btc-usdm-rsi-30-70-2025",
)

In [ ]:
summary = result.summary

{
    "bars": result.report.run.bar_count,
    "fills": summary.total_fills,
    "closed_trades": summary.total_trades,
    "final_equity": summary.final_equity,
    "total_return": summary.total_return,
    "max_drawdown": summary.max_drawdown,
    "win_rate": summary.win_rate,
}

In [ ]:
fig = result.plot()
fig